In [1]:
from pathlib import Path

import pickle
import pynapple as nap

import numpy as np
# from notebooks.data_synchronization_barcode import tsync_barcode
%matplotlib widget

import matplotlib.pyplot as plt

from ipywidgets import interact
import ipywidgets as widgets

from tqdm.notebook import tqdm

import pandas as pd

# import seaborn as sns
# custom_params = {"axes.spines.right": False, "axes.spines.top": False, "figure.figsize": (8, 4)}
# sns.set_context("paper")
# sns.set_theme(style="ticks", palette="colorblind", font_scale=1.3, rc=custom_params)


In [2]:
root_path = Path("/data/ofl_2p/20251118")

project = nap.load_folder(root_path)
print(project)

📂 20251118
├── 📂 00001
├── 📂 preprocessing_results
├── 📂 s3d-00001split
├── 📂 s3d-xy_1000
└── 📂 split_tifs

In [3]:
session = project["preprocessing_results"]["behavior_sync"]
session.view

📂 behavior_sync
├── 📂 behavior
│   ├── Valve_Toggle.npz        |        Tsd
│   ├── Reward_Tile.npz         |        Tsd
│   ├── Nature_Environment.npz  |        Tsd
│   ├── New_Trial.npz   |        Tsd
│   ├── Test_Shock.npz  |        Tsd
│   ├── Shock_Grid.npz  |        Tsd
│   ├── longVar.npz     |        TsdFrame
│   ├── Lick_Detection.npz      |        Tsd
│   ├── transmitTS.npz  |        Tsd
│   ├── OFL_Experiment.npz      |        Tsd
│   ├── Pin_Sync_LED.npz        |        Tsd
│   ├── Halloween_Environment.npz       |        Tsd
│   ├── licks.npz       |        Tsd
│   ├── uncorrected_distance.npz        |        Tsd
│   ├── Breathing_Sensor.npz    |        Tsd
│   ├── Tunnel_2.npz    |        Tsd
│   ├── packetNums.npz  |        Tsd
│   ├── Finish_Trial.npz        |        Tsd
│   ├── Trigger_C.npz   |        Tsd
│   ├── Scanner_Frame_Clock_Input.npz   |        Tsd
│   ├── startTS.npz     |        Tsd
│   ├── Wheel_Encoder_B.npz     |        Tsd
│   ├── Tunnel_1.npz    |        Tsd
│   ├── GND.npz         |        Tsd
│   ├── Ephys_Trigger.npz       |        Tsd
│   ├── Wheel_Encoder_Z_check.npz       |        Tsd
│   ├── City_Environment.npz    |        Tsd
│   ├── corrected_distance.npz  |        Tsd
│   ├── Pre_Shock_Experiment.npz        |        Tsd
│   ├── Barcode_Scanner.npz     |        Tsd
│   ├── Wheel_Encoder_A.npz     |        Tsd
│   └── Wheel_Encoder_X_check.npz       |        Tsd
├── 20251118-152602_925_barcode_data.npz    |        npz
├── 20251118-152602_925_frames_time_idx.npz         |        Tsd
├── Fneu_sync.npz   |        TsdFrame
├── F_sync.npz      |        TsdFrame
└── spks_sync.npz   |        TsdFrame

In [4]:
sync_stats_file = root_path / "preprocessing_results/behavior_sync" / "20251118-152602_925_behavior_sync_stats.pkl"
with open(sync_stats_file, "rb") as f:
    sync_stats = pickle.load(f)

In [5]:
sync_stats

{'session': '20251118-152602_925',
 'max_ts_gap': 121000,
 'gap_locations': array([6.68232993e+08, 1.38604699e+09]),
 'has_barcode': False}

In [6]:
def extract_rewards(r: nap.IntervalSet):
    rew_tsd = r.threshold(0.5)
    rew_is = rew_tsd.time_support.merge_close_intervals(threshold=0.2)
    return nap.Ts(rew_is.start)

In [7]:
rewards = extract_rewards(session["behavior"]["Valve_Toggle"])

In [10]:
spks = session['spks_sync']

In [14]:
spks

Time (s)      0         1          2         3         4  ...
----------  ---  --------  ---------  --------  --------  -----
29.087993     0    0         0          0        0        ...
29.187993     0  204.631     0         29.3449  55.2906   ...
29.287993     0    0       182.137    150.363    0        ...
29.387993     0    0         6.01458  148.748    0        ...
29.487993     0   17.3301    9.95168   88.2017   0        ...
29.587993     0   85.4159  218.468      0        0        ...
29.688993     0    0         0          0        0        ...
...                                                       ...
667.539993    0   20.7389    0          0        0        ...
667.639993    0    0         0          0        0        ...
667.739993    0    0        51.4083     0        9.0274   ...
667.839993    0    0        72.8112    20.8067   7.19767  ...
667.940993    0    0         0          0        0        ...
668.040993    0    0         0          0        0        ...
668.14

In [11]:
def spks_as_tsgroup(threshold: float, spks: nap.TsdFrame):
    t = spks.t
    n_cells = spks.shape[1]
    ts_dict = {i: nap.Ts(t[spks[:, i] > threshold]) for i in range(n_cells)}
    return nap.TsGroup(ts_dict)

In [12]:
spikes = spks_as_tsgroup(100, spks)

/home/battaglia/miniforge3/envs/ofl-2p-analysis/lib/python3.11/site-packages/pynapple/core/base_class.py:50: UserWarning: Some epochs have no duration
  self.time_support = IntervalSet(start=self.index[0], end=self.index[-1])
/home/battaglia/miniforge3/envs/ofl-2p-analysis/lib/python3.11/site-packages/pynapple/core/base_class.py:52: RuntimeWarning: divide by zero encountered in scalar divide
  self.rate = self.index.shape[0] / np.sum(


In [13]:
peth = nap.compute_perievent(data=spikes, events=rewards, window=(-2, 5))
peth

{0:   Index    rate    events
 -------  ------  --------
       0       0    517.96,
 1:   Index     rate    events
 -------  -------  --------
       0  0.28571    517.96,
 2:   Index    rate    events
 -------  ------  --------
       0       0    517.96,
 3:   Index    rate    events
 -------  ------  --------
       0       0    517.96,
 4:   Index    rate    events
 -------  ------  --------
       0       0    517.96,
 5:   Index     rate    events
 -------  -------  --------
       0  0.14286    517.96,
 6:   Index    rate    events
 -------  ------  --------
       0       0    517.96,
 7:   Index    rate    events
 -------  ------  --------
       0       0    517.96,
 8:   Index    rate    events
 -------  ------  --------
       0       0    517.96,
 9:   Index    rate    events
 -------  ------  --------
       0       0    517.96,
 10:   Index    rate    events
 -------  ------  --------
       0       0    517.96,
 11:   Index    rate    events
 -------  ------  --------


In [ ]:
def plot_peth(unit_peth, unit_peth_counts, ax_mean, ax_spikes, color=None):
    mean = np.mean(unit_peth_counts / bin_size, axis=1)
    ax_mean.plot(mean, color=color)
    ax_mean.set_ylabel("spikes/s")
    ax_mean.axvline(0.0, color="gray", linestyle="--")
    ax_spikes.plot(unit_peth.to_tsd(), "|", markersize=5, color=color)
    ax_mean.set_xlabel("time from event (s)")
    ax_spikes.set_ylabel("event")
    ax_spikes.axvline(0.0, color="gray", linestyle="--")


fig, (ax_spikes, ax_mean) = plt.subplots(
    2, 1, sharex=True, height_ratios=[1, 0.3], figsize=(6, 6), gridspec_kw={"hspace": 0.3}
)
plot_peth(peth, peth_counts, ax_mean, ax_spikes)
ax_spikes.set_title("peri = nap.compute_perievent(spikes, events)")
ax_mean.set_title("peri.count(bin_size) / bin_size")

In [ ]:
fig, axs = plt.subplots(
    2,
    len(tsgroup),
    sharey="row",
    sharex=True,
    height_ratios=[0.3, 1.0],
    figsize=(15, 8),
)
fig.suptitle("nap.compute_perievent(tsgroup, stimuli, (-0.1, 0.4)", y=1.02)
for i, (unit, unit_axs) in enumerate(zip(tsgroup, axs.T)):
    plot_peth(peth[unit], peth[unit].count(bin_size), *unit_axs, color=plt.cm.tab10(i))
    unit_axs[0].set_title(f"unit {unit}")

In [ ]:
(fig, ax) = plt.subplots(figsize=(10, 4))

ax.plot(spks[:,421])
ax.plot(spikes[421].fillna(0), '.', label="cell 421")

In [ ]:
rewards

In [ ]:

r = session["behavior"]["Valve_Toggle"]

In [ ]:
r